**Module**: Data 9910 – Working with Data

**Lecturer**: Lucas Rizzo



**Student Details**

- student name = eamonn kelly
- student number = D24127620

## Introduction


### Project Objective

Project Question = *Does the Sun have a twin or a closely related cousin?*

Yhe goal of this project is to see can we determine if there another star in the universe that is identical or similar to our Sun, that we know of and have measurements, metrics for.


### Data Sources
We will sue astronmical data from a variety of different public sources. 

1) Stellar Hosts - nasa - one row per star.
    - contains all knwon exoplanet host stars and may parameter values for those stars.
2) pscompars (planetary system composite parameters)= nasa
    - one row per planet contains star details which are useful so takign and will mergeinto stellar host data
4) GAIA - DR3 - for around 1.46 billion (1.46 109) sources >> https://www.cosmos.esa.int/web/gaia/dr3 
    - Sources in the Gaia Catalogue are all identified through the Gaia Source Identifier, i.e., the source_id field in the various tables in the Gaia Archive > is from Dr3 is different to DR2 etc
   -  3a) 3) I/355/paramsp (astrphysical_parameters) - ESA (GAIA) - (1.5 million rows)
    https://www.aanda.org/articles/aa/full_html/2023/06/aa43688-22/aa43688-22.html 
    - this is a subset of calculated data from the DR3 dataset it is a table within the DR3 dataset
    - this is what we will use
4) LAMOST (Large Sky Area Multi object Fiber Spectroscope) - China - https://www.lamost.org/lmusers/user/
    - latest data DR13 requires approval and login
    - DR10 is downloadable - so using that


### Issues encountered
- had problems with gaia IDs and bigint data types, gett exponential numbers,  conversion issues, losing digits. decided to use strings for those 19 digit ID numbers. theyh are also identifies, labels, so not used mathematically, so no calculations use dno them as such string is fine.

- FIT Files as a data source file type - come as indiviaul files, have to be merged standane sets fo data.

- accounts
    - ESA > https://psaftp.esac.esa.int/WebInterface/login.html
        -username = ekelly
        - pwd = SlieveMor01!

    - GAIA - catalog sizes  > https://cdn.gea.esac.esa.int/Gaia/gdr3/_catalogue_sizes.txt
    - LAMOST - 

## Part 1

Import the piece/s of data and perform any cleaning and merging to produce a final dataframe

In [112]:
import pandas as pd
import numpy as np
import kagglehub
import re
# esascientific community package for accesing and downloadign data
# https://www.astropy.org/
from astroquery.gaia import Gaia
from astropy.table import Table
# french mirror of thre ea gaia data sets
from astroquery.vizier import Vizier

import requests
import time
from bs4 import BeautifulSoup
import json
from datetime import datetime
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
#defining global variable path to files and data, it is called thropughout modify this path to match your local path
path = './'

#### 1) Dataset - NASA >  Stellar Hosts

Data set taken from https://exoplanetarchive.ipac.caltech.edu/ 

We will down lai the 'Stellar Hosts' dataset. This dataset is 1 row per star. It is determined by the search for exoplanets, sok each star will have 0 to > 0 number of planets orbitting it.

##### 1a) Download stallar hosts dataset

In [ ]:
# after a bit of playing around tihe the url format we get the url for csv format for the entire stellar hosts dataset
# we download in csv format
# This uses their preferred TAP API, as adql query to directly query the database, and we get it in a clean csv.
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+stellarhosts&format=csv"
stellar_host_df = pd.read_csv(url)

stellar_host_df.shape

# create a lcoal copy in csv as had issue with server going offline and access data. so save lcoal copy in case needed
# wil put a time stampe on the filename for uniqueness
datetime_stamp = datetime.now().strftime("%d%m%y_%H")
sh_csv_file_name = f"{path}stellarhosts_{datetime_stamp}.csv"
stellar_host_df.to_csv(sh_csv_file_name, index=False)
print(f"saved {sh_csv_file_name}")

# NOTE th is nto one row per star >> all sceintific measurements are kept and sources are different, so needs to be filered....
# may add update a paramater, i.e. update mass of a star , added in a new row, old row kept...

saved ./stellarhosts_111225.csv


In [166]:
# check downloadded fine and get feel for shape of data downloaded
# stellar_host_df.head()

stellar_host_df.shape

(46870, 136)

#### 1b) obtain GAIA IDs for use in other downloads

Need to get the uniqwue values for GAIA_IDs. We will sue those to download other datasets. Other datasets are to big, in TBs in siuze, and this will allow us filter just what we need, reduce downlaod time and make ti more manageable

In [15]:
# get colimns containign the string 'gaia' from the dataframe
# gaia IDs are the column we will merge against and also pivit around across daatsets. This is a unique identifier used by the ESA
# we willa also use gaia ID to filter data we down load from other sources, to limit file size to only relevant data/rows.
gaia_cols = [col for col in stellar_host_df.columns if'gaia' in col.lower()]

print(f"gaia columns  : {gaia_cols}")

gaia columns  : ['sy_gaiamag', 'sy_gaiamagerr1', 'sy_gaiamagerr2', 'gaia_dr2_id', 'gaia_dr3_id']


Check the data format. we will leave as astr object types as convertign to int rounds and we havhe huge 19 digit numbers, we can do string matches across datasets

As per gudielines on EAA site there is no direct relationship between GAIA DR2 IDs and DR3 IDs

Sources in the Gaia Catalogue are all identified through the Gaia Source Identifier, i.e., the source_id field in the various tables in the Gaia Archive. the source list for Gaia DR3 should be treated as independent from Gaia DR2 and from Gaia DR1. With each new Gaia data release, the source list is becoming progressively more stable

Refer to [Gaia Data Release 3 (Gaia DR3)](https://www.cosmos.esa.int/web/gaia/dr3) for details from above amnd futher details re the data set.


In [16]:
stellar_host_df['gaia_dr3_id'].head(10)

0    Gaia DR3 2128190453050802048
1    Gaia DR3 2105930840143687680
2    Gaia DR3 2077595394707557120
3    Gaia DR3 2085724496490595584
4    Gaia DR3 2129158435598210816
5    Gaia DR3 2086414268238649984
6    Gaia DR3 2052582432887909376
7    Gaia DR3 2052942900903300352
8    Gaia DR3 2130211080541825024
9    Gaia DR3 2128079230577866880
Name: gaia_dr3_id, dtype: object

craete a list object into which we'll put these strings, we'll refer to this list when downlaoding other data sources

strip the text and number not needed, so onyl get 19 digit number at the end of string, use regex expressino to filter the cell values

i.e. want to get the GAIA ID numbers minus the prefix strings

Problem
need to match only the gaia IDs numbers so can downlaod only the data we wnat from gaia downlaod
DR3 and DR4 are nto 100% identical m and this is as expected as ID get updated as data is updated sources are split, uydpated data changes thigns etc so it loks like our data is intact and we have ths striped gaiaa ID

In [ ]:
# we'll create a copy to work in, keepign our roginal in tac if we need tok revert to it at some stage
stellar_host_df_2 = stellar_host_df.copy()

# http://regex101.com
# $ start at the end of thestring, only match the last part,
# smatch 10 to 20 digits in a row, ignore the space, so we don't get the 2 from DR2, so contiguous, between 10 and 20 characters and at the end of the string 
# stop at space
stellar_host_df_2['gaia_dr3_num'] = stellar_host_df_2['gaia_dr3_id'].astype(str).str.extract(r'(\d{10,20})', expand=False)

print(stellar_host_df_2['gaia_dr3_num'].head(10))

# stellar_host_df_2['dr2_equals_dr3'] = stellar_host_df_2['dr2_num'] == stellar_host_df_2['dr3_num']

# stellar_host_df_2 = stellar_host_df_2.dropna(subset=['dr3_num'])stellar_host_df_2['dr2_equals_dr3'].value_counts()

0    2128190453050802048
1    2105930840143687680
2    2077595394707557120
3    2085724496490595584
4    2129158435598210816
5    2086414268238649984
6    2052582432887909376
7    2052942900903300352
8    2130211080541825024
9    2128079230577866880
Name: gaia_dr3_num, dtype: object


In [26]:
# earlier shape was (46870, 136) - ti si now (46870, 137) which is as expected and we have an additional column for our gaia_dr3)num values
stellar_host_df_2.shape

(46870, 137)

Check to see fi cell values are as we expect, identify ant problematoc cell data

In [33]:
# check to see if gaia Ids are as we expect and what ros contain data we are not expecting and are problematic
# Convert to string for consistent checks
col = stellar_host_df_2['gaia_dr3_num'].astype(str)

diagnostics = {
    "total_rows": len(col),

    # Missing or NA-like
    "is_na": col.isna().sum(),
    "empty_string": (col.str.strip() == "").sum(),
    "literal_nan_or_none": col.str.lower().isin(["nan", "none"]).sum(),

    # Invalid characters (non-digits)
    "contains_nondigits": col.str.contains(r"\D", regex=True).sum(),

    # Has a valid digit sequence (extractable)
    "contains_digit_block_10_20": col.str.contains(r"\d{10,20}", regex=True).sum(),

    # Fullmatch valid Gaia ID (exact digits only)
    "valid_fullmatch": col.str.fullmatch(r"\d{10,20}").sum(),

    # Wrong length but digits-only
    "digits_only_wrong_length": (
        col.str.fullmatch(r"\d+").fillna(False) &
        ~col.str.fullmatch(r"\d{10,20}")
    ).sum(),
}

pd.Series(diagnostics)


total_rows                    46870
is_na                             0
empty_string                      0
literal_nan_or_none            1738
contains_nondigits             1738
contains_digit_block_10_20    45132
valid_fullmatch               45132
digits_only_wrong_length          0
dtype: int64

Remove problematic cells, and ensure shape # of rows matches shape value for rows continaing correct data structure as per earlier diagnostic

In [34]:
# remove the probllematic gaia ID rows

sh_clean_gaiaid_df = stellar_host_df_2[stellar_host_df_2['gaia_dr3_num'].astype(str).str.fullmatch(r'\d{10,20}')].copy()

#checl rows have been removed
sh_clean_gaiaid_df.shape

# sh_clean_gaia_df['dr3_num'].head()

(45132, 137)

re run disgnostics to verify the changes/deletions - verufying the new dataframe contains only valid gaia IDs and  - and  new df shape is 45132, 137

In [35]:
# check to see if gaia Ids are as we expect and what ros contain data we are not expecting and are problematic
# Convert to string for consistent checks
col = sh_clean_gaiaid_df['gaia_dr3_num'].astype(str)

diagnostics = {
    "total_rows": len(col),

    # Missing or NA-like
    "is_na": col.isna().sum(),
    "empty_string": (col.str.strip() == "").sum(),
    "literal_nan_or_none": col.str.lower().isin(["nan", "none"]).sum(),

    # Invalid characters (non-digits)
    "contains_nondigits": col.str.contains(r"\D", regex=True).sum(),

    # Has a valid digit sequence (extractable)
    "contains_digit_block_10_20": col.str.contains(r"\d{10,20}", regex=True).sum(),

    # Fullmatch valid Gaia ID (exact digits only)
    "valid_fullmatch": col.str.fullmatch(r"\d{10,20}").sum(),

    # Wrong length but digits-only
    "digits_only_wrong_length": (
        col.str.fullmatch(r"\d+").fillna(False) &
        ~col.str.fullmatch(r"\d{10,20}")
    ).sum(),
}

pd.Series(diagnostics)


total_rows                    45132
is_na                             0
empty_string                      0
literal_nan_or_none               0
contains_nondigits                0
contains_digit_block_10_20    45132
valid_fullmatch               45132
digits_only_wrong_length          0
dtype: int64

Craete two new list objects one for full gaia ID and one for justnumerc gaia ID, wsome data sources will filter off only numeric data and some will filter off fill ID, so just creatign two list objects and we cna use either fir dat download as we need.

In [43]:

sh_clean_gaiaid_fullid_list = sh_clean_gaiaid_df['gaia_dr3_id'].astype(str).tolist()

sh_clean_gaiaid_fullid_list[:3]

['Gaia DR3 2128190453050802048',
 'Gaia DR3 2105930840143687680',
 'Gaia DR3 2077595394707557120']

In [93]:
sh_clean_gaiaid_num_list = sh_clean_gaiaid_df['gaia_dr3_num'].astype(str).tolist()

sh_clean_gaiaid_num_list [:3]

['2128190453050802048', '2105930840143687680', '2077595394707557120']

#### 2) PSCompars (pscompars (planetary system composite parameters)= nasa)

Data set taken from https://exoplanetarchive.ipac.caltech.edu/ 


In [45]:
# downlaod using ADQL and the tap server
# we'll save as csv aswell as have had problems with downtime and maintenance on  servers, and them not being available, so as a backup
# this is the full dataset
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+pscomppars&format=csv"
pscompars_df = pd.read_csv(url)

# use dat time stanpe to give us unique id for the csv, so we don't overwirte earlier downlaods
datetime_stamp = datetime.now().strftime("%d%m%y_%H")
pscompars_csv_file_name = f"{path}pscompars_{datetime_stamp}.csv"
pscompars_df.to_csv(pscompars_csv_file_name, index=False)
print(f"saved {pscompars_csv_file_name}")


print(pscompars_df.shape)
#pscompars_df.cols

saved ./pscompars_111225_06.csv
(6053, 683)


In [165]:
pscompars_df.shape

(6053, 683)

#### 3) GAIA DR3 Astrophysical parameters

there ar approx 400 million + rows in the dr3 dataset, we ar looking to down load just the ap table data form the dr3 dataset. so we have to filter against our gaia_dr3_id list which we created earlier from our stellar host data set.

We will match against the ap table column 'source _id' which  uses 'long' datatype objects for source id, so 64 bit integer,  we get errors if we match the full gaia id, as such will match against the source id with numeric string from our earlier created list.

A list of dataset column data types is available here > https://gaia.aip.de/metadata/gaiadr3/astrophysical_parameters/ 

Check to  see how many rows of data are in the Gaia DR3 Astrophysical paramaters table dataset before we download i.e. just the ap table, not the whole DR3 dataset i.e. approx 1.5 billion



In [167]:
# sql query we will run
# tripple quotes to allow us use multi line
query_gaia_ap_row_count_query = """
SELECT COUNT(*)
FROM gaiadr3.astrophysical_parameters
"""

# usign the astroquery package
# returns a table
job = Gaia.launch_job_async(query_gaia_ap_row_count_query)
count_tbl = job.get_results()
print(f"number of rows in gaia DR3 AP dataset table  = ", count_tbl[0][0] )

INFO: Query finished. [astroquery.utils.tap.core]
number of rows in gaia DR3 AP dataset table  =  1590932717


Lets see how manhy colmns are in the AP dataset before download, i.e. again just the ap table, not the whole DR3 dataset i.e. 225 columns

In [168]:
# sql query we will run
# tripple quotes to allow us use multi line
query_gaia_ap_col_count_query = """
SELECT COUNT(*)
FROM TAP_SCHEMA.columns
WHERE table_name = 'gaiadr3.astrophysical_parameters'
"""

# usign the astroquery package
# returns a table
job = Gaia.launch_job_async(query_gaia_ap_col_count_query)
count_tbl = job.get_results()
print(f"number of columns in gaia DR3 AP dataset table  = ", count_tbl[0][0])

INFO: Query finished. [astroquery.utils.tap.core]
number of columns in gaia DR3 AP dataset table  =  226


Check to verify the data type for the' source_id' columnn in the dataset we want to download is a we expect. It is long, so 64 bit integer, some values are 19 digits long.

In [169]:
# sql query we will run
# tripple quotes to allow us use multi line
query_ap_col_dtype = """
SELECT column_name, datatype
FROM TAP_SCHEMA.columns
WHERE table_name = 'gaiadr3.astrophysical_parameters'
"""

# usign the astroquery package
job = Gaia.launch_job_async(query_ap_col_dtype)
result = job.get_results()
print(result[:10])

INFO: Query finished. [astroquery.utils.tap.core]
          column_name            datatype
-------------------------------- --------
               nfe_gspspec_upper    float
              nfe_gspspec_nlines      int
         nfe_gspspec_linescatter    float
                     solution_id     long
                       source_id     long
    classprob_dsc_combmod_quasar    float
    classprob_dsc_combmod_galaxy    float
      classprob_dsc_combmod_star    float
classprob_dsc_combmod_whitedwarf    float
classprob_dsc_combmod_binarystar    float


function to check if tap serevr is online as have ahd problems wit it being offline for periods.

In [171]:
# function to check if gaia tap server  is online...
# gettgin errors and nto sur eif onlien or nto, so builing separate fnction that can call during download
def check_gaia_tap():
    url = "https://gea.esac.esa.int/tap-server/tap/availability"
    try:
        r = requests.get(url, timeout=5)
        if r.status_code == 200 and "available" in r.text.lower():
            print("Gaia TAP service is online and available")
            return True
        else:
            print("Gaia TAP service reachable but not fully available")
            return False
    except Exception as e:
        print("Gaia TAP service is NOT online and NOT available", e)
        return False

Download the GAIA DR3 Astrophysical parameters. determine appriopriate columns and data to download.
we will focus mostly on the gsp -Phot data
- GSP-Phot (Generalized Stellar Parametrizer) - https://www.mpia.de/gaia/projects/gsp
- ESP-HS (Extended Stellar Parameteriser for Hot Stars)
- FLAME (FrAscati Luminosity And Mass Estimator)  - evolutionary parameters for stars

In [172]:
# function to download gaia astrohysycal parameter datsset and only get rows which match gaia dr3 ids in our list
# the iput is how many ids for each query i.e. batch_size
# downlaoads were hangin so dong it in packates helped to prevent tat, and added to function so cna repeat it
# https://astroquery.readthedocs.io/en/latest/_modules/astroquery/mast/observations.html#ObservationsClass.get_product_list_async

def download_gaia_dr3_ap_data(gaia_ids, batch_size=1000):
    # convert the gaia ids into strings so wel take in out gaia_ids form our gaia id list
    gaia_ids = [str(x) for x in gaia_ids] 

    # the length of our list of ids, total nujmber
    n = len(gaia_ids)
    # need to split our iD list into batches
    # set a range value of from 0 to len to dthe delta or batch size
    # for each value of i we slice in that range from i to 1 plus batch size...so we have a list of ids in that range 
    batches = [gaia_ids[i:i + batch_size] for i in range(0, n, batch_size)]
    # whats happenign
    print(f"Downloading gaia dr3 ap data in {len(batches)} batches...")

    # create empty list for storign dataframe output
    # we'll combine these later
    ap_dataframe_list = []

    # for loop to go througn each batch and run the query, so we get ti in batches
    # idx is the batch number, for each batch number, starting at 1 and nto zero
    for idx, batch in enumerate(batches, 1):
        # whats happening, is it hung
        print(f"\nbatch {idx}/{len(batches)}...")

        # convert list of ids into comma-separated string as we need that format in the sql query
        # otherwise it fails, we can't run the full gaia id in the sql query, need to use number
        id_str = ",".join(batch)

        # sql query specify the spurce and incldgijn where conditin of the gaia id
        # the source_id is the unique identifier int eh ap dataset, 
        # as from earlier there are 228 columns, so we identified pticular columns we think wil eb of use, 
        # we sue triple quotes to allow use use multi ilne string

        # there are different sources for the data also or how they were derived such as ASP-HS, FLAME, GSP-Phot
        # lots of chemical composition data - nitrogern, iron, sulphur, neodymium, magnesium, titanium, calcium, silocon and pecentile calculations, we will tno be takign any of those, as its beyon scope and time available
        query = f"""
        SELECT source_id, teff_gspphot, logg_gspphot, mg_gspphot, mh_gspphot, radius_gspphot, distance_gspphot, lum_flame, mass_flame, age_flame, evolstage_flame, ag_gspphot
        FROM gaiadr3.astrophysical_parameters
        WHERE source_id IN ({id_str})
        """

        # the actual query code is executed
        # send sql query to gaia data spurce, calls the query and gets a result downlaoded into 
        # https://astroquery.readthedocs.io/en/latest/api/astroquery.gaia.GaiaClass.html
        # use asyncronous job as submites to queue on gaia server and helps prevent hanging, prevent timeouts
        job = Gaia.launch_job_async(query)
        result = job.get_results()

        # convetr the data to a df
        inloop_ap_df = result.to_pandas()
        # stores ina  list and appemnds to previous list in each iteration
        ap_dataframe_list.append(inloop_ap_df)

        # getting hangs on download so adding pause at end of each loop to separate them to stop rate limitiing
        time.sleep(0.3)   

    # Combine all returned rows
    final_df = pd.concat(ap_dataframe_list, ignore_index=True)
    return final_df



In [173]:
# cll the function
# # call the is online function
# if check_gaia_tap():
print("Downloading Gaia DR3 ap data rows  - downlaoding only rows that match gaia id list values")

# call the download function with the gaia dr3 id num liat object, 
gaia_dr3_ap_df = download_gaia_dr3_ap_data(sh_clean_gaiaid_num_list)


# # create a lcoal copy in csv as had issue with server going offline and access data. so save lcoal copy in case needed
# # wil put a time stampe on the filename for uniqueness
# datetime_stamp = datetime.now().strftime("%d%m%y_%H")
# gaia_dr3_ap_data_csv_file_name = f"{path}gaia_dr3_ap_{datetime_stamp}.csv"
# gaia_ap_subset.to_csv(gaia_dr3_ap_data_csv_file_name, index=False)
# print(f"saved {gaia_dr3_ap_data_csv_file_name}")


# else:
#     print("gaia tap service is offline — stopping")

# create a lcoal copy in csv as had issue with server going offline and access data. so save lcoal copy in case needed
# wil put a time stampe on the filename for uniqueness
datetime_stamp = datetime.now().strftime("%d%m%y_%H")
gaia_dr3_ap_data_csv_file_name = f"{path}gaia_dr3_ap_{datetime_stamp}.csv"
gaia_ap_subset.to_csv(gaia_dr3_ap_data_csv_file_name, index=False)
print(f"saved {gaia_dr3_ap_data_csv_file_name}")


print("Gaia DR3 AP dataframe shape  = ", gaia_dr3_ap_df.shape)


batch 1/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 2/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 3/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 4/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 5/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 6/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 7/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 8/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 9/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 10/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 11/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 12/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 13/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 14/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 15/46...
INFO: Query finished. [astroquery.utils.tap.core]

batch 16/46...
INF

In [84]:
ids = sh_clean_gaiaid_num_list[:1500]
print(len(",".join(map(str, ids))))


29912


#### 4) LAMOST (Latge Sky Area Multi-Object Fiber Spectroscopic Telescope)

detaisl available from her > https://www.lamost.org/public/?locale=en 

DR13 s tye latest set of data but you need lgigin IDs and approval to downlaod ans use, hence we will sue an older set pf daya DR11.

https://www.lamost.org/dr11/v2.0/catalogue

there are options to downlaod via FTP or european mirrors btu accoutns are required for login, am still aiting for approval, column names have ben altered in some mirrors also. The other optji is tok manually download. I have chosent o manually down catalogue and then filter locally. 

I have queried to see colimn mames >> https://www.lamost.org/dr11/v2.0/table/stellar and ti has a gaia_source_id column whicj we cna merge on.

We wil download the LAMOST MRS Parameter Catalogue. This is approx 1GB in size and download took approximately 20 mins. we re-crate a smaller filtered csv off that full csv catalogue download. 

I Decided to manually download the dataset > dr11_v2.0_MRS_stellar.fits > dr11_v2.0_MRS_stellar.fits.gz
- It is approx 1 GB in size when extracted - too big to include in assignmnt hand off. So in case ipynb file needs to be run, I will create smaller fit file for inclusiong off this 2gb fit file and will include thatyin the hand off.
= extract the tar.gz file
- build dataframe from extracted fit file
- it has a shape of (7898024, 45)

we will use FIT fie format to try help avoin d any corruption when covertign from csv



In [ ]:
# including code here but commenting out as its not intended ot be run again. csv has already been created off manually downlaoded 
# lamost fits  file
# master dataframe has a shape of >> (7898024, 45)

# Load FITS file
lamost_fits_file = f"{path}dr11_v2.0_MRS_stellar.fits"   
lamost_fits_tbl = Table.read(lamost_fits_file)

# Convert entire table to a pandas DataFrame
lamost_full_stellar_cat_fits_df = lamost_fits_tbl.to_pandas()

print(lammost_full_stellar_cat_fits_df.shape)
lamost_full_stellar_cat_fits_df.head()


(7898024, 45)


,mobsid,obsid,uid,gp_id,designation,obsdate,lmjd,mjd,planid,spid,...,ca_fe,ti_fe,cr_fe,ni_fe,cu_fe,alpha_m_lasp,alpha_m_lasp_err,moon_angle,lunardate,moon_flg
0,b'589102005B',589102005,b'G13674228956942',2007175828177988096,b'J225320.50+574949.7',b'2017-09-28',58025,58024,b'apogee_field0101',2,...,-0.09265,-9999.00000,-9999.00000,0.02812,-9999.00000,0.030230,0.016372,94.5,9,b'0'
1,b'589102011B',589102011,b'G13674230881236',2007179710828415232,b'J225236.56+575228.9',b'2017-09-28',58025,58024,b'apogee_field0101',2,...,0.04382,-0.01623,-0.02655,0.01371,0.02801,0.013985,0.038074,94.5,9,b'0'
2,b'589102021B',589102021,b'G13674222921556',2007196276531913216,b'J225237.17+580219.0',b'2017-09-28',58025,58024,b'apogee_field0101',2,...,-0.01571,-9999.00000,-9999.00000,-0.01356,-9999.00000,-9999.000000,-9999.000000,94.5,9,b'0'
3,b'589102027B',589102027,b'G13673784720856',2007438409604543104,b'J224808.02+582034.5',b'2017-09-28',58025,58024,b'apogee_field0101',2,...,0.00009,-9999.00000,-9999.00000,-0.04321,-9999.00000,-0.071275,0.041089,94.5,9,b'0'
4,b'589102029B',589102029,b'G13674121929935',2007452424071586560,b'J224926.62+582734.1',b'2017-09-28',58025,58024,b'apogee_field0101',2,...,0.13706,-0.11110,0.02287,0.03556,0.20856,-9999.000000,-9999.000000,94.5,9,b'0'


In [146]:
# gives a shape of (7898024, 45)
lamost_col_dtype_info_df = (
    lamost_full_stellar_cat_fits_df
    .dtypes
    .reset_index()
    .rename(columns={"index": "column", 0: "dtype"})
)



lamost_col_dtype_info_dict = dict(zip(lamost_col_dtype_info_df["column"], lamost_col_dtype_info_df["dtype"]))


lamost_col_dtype_info_dict

{'mobsid': dtype('O'),
 'obsid': dtype('int32'),
 'uid': dtype('O'),
 'gp_id': dtype('int64'),
 'designation': dtype('O'),
 'obsdate': dtype('O'),
 'lmjd': dtype('int32'),
 'mjd': dtype('int32'),
 'planid': dtype('O'),
 'spid': dtype('int16'),
 'fiberid': dtype('int16'),
 'lmjm': dtype('int32'),
 'band': dtype('O'),
 'ra_obs': dtype('float64'),
 'dec_obs': dtype('float64'),
 'snr': dtype('float32'),
 'gaia_source_id': dtype('O'),
 'gaia_g_mean_mag': dtype('float32'),
 'gaia_bp_mean_mag': dtype('float32'),
 'gaia_rp_mean_mag': dtype('float32'),
 'fibertype': dtype('O'),
 'offsets': dtype('int16'),
 'offsets_v': dtype('float32'),
 'ra': dtype('float64'),
 'dec': dtype('float64'),
 'teff_lasp': dtype('float32'),
 'teff_lasp_err': dtype('float32'),
 'logg_lasp': dtype('float32'),
 'logg_lasp_err': dtype('float32'),
 'feh_lasp': dtype('float32'),
 'feh_lasp_err': dtype('float32'),
 'vsini_lasp': dtype('float32'),
 'vsini_lasp_err': dtype('float32'),
 'rv_b0': dtype('float32'),
 'rv_b0_err':

In [148]:
# Filter for GAIS DR3 IDs  - we onyl keep those entries that we can match across datasets
# this will be our workign dataframe - which we will clan properly then.
# was using a loop but was takign too long....

# m,ake sure both dtypes in the id list w created earlier and the downlaoded lamost catalogue are both str dtyeps
lamost_full_stellar_cat_fits_df["gaia_source_id"] = (lamost_full_stellar_cat_fits_df["gaia_source_id"].astype(str).str.strip().replace(['None', 'nan', 'NaN', '', ',' '.'], pd.NA))

# 
sh_clean_gaiaid_num_list = [str(x) for x in sh_clean_gaiaid_num_list]



In [ ]:
lamost_matched_df = lamost_full_stellar_cat_fits_df[lamost_full_stellar_cat_fits_df["gaia_source_id"].isin(sh_clean_gaiaid_num_list)]

print("Matches:", lamost_matched_df.shape)

# DR10 = Matches: (838, 76)
# DR11 = Matches: (968, 76)

Matches: (968, 76)


968 star matches across the GAIA DR3 and LAMOST DR11 datasets seems like a low number. We validate the results below again, to double check our data.

- GAIA DR3 has 45K+ unique stars
- LAMOST DR11 has 2.5 million + unique stars
- It may be explainable due to different parts of the sky being observed i.e. China vs Europe and the overlap between those.
- Another explanation may eb the LAMOST dataset ot being the latest, perhaps DR13 dataset being the latest and a bigger dataset, would have more matches
-  It would require further investigation to determine, which is beyond the scope of the time we have available for this assignemnt at the moment.

In [150]:
print(f" Number of gaia dr3 ids we have in our list to check against =  {len(sh_clean_gaiaid_num_list)}")

print(f" Number of unique non empty lamost ids that we are matchig againsty = {lamost_full_stellar_cat_fits_df["gaia_source_id"].notna().sum()}")

print(f" Matchign Stars across the GAIA DR£ and LAMOST DR11 datasets =  {len(set(sh_clean_gaiaid_num_list) & set(lamost_full_stellar_cat_fits_df["gaia_source_id"].dropna()))}")


 Number of gaia dr3 ids we have in our list to check against =  45132
 Number of unique non empty lamost ids that we are matchig againsty = 2594070
 Matchign Stars across the GAIA DR£ and LAMOST DR11 datasets =  294


we'll now crate a new lamost dr11 dataframe that contains only stars which have a match gaia id 

In [ ]:
# filter the dr11 dataframe for gaia ids matches
lamost_gaiaid_matched_dr11_df = lamost_full_stellar_cat_fits_df[lamost_full_stellar_cat_fits_df["gaia_source_id"].isin(sh_clean_gaiaid_num_list)].copy()

print(f" new lamost gaisid matched dr11 dataframe shape =  {lamost_gaiaid_matched_dr11_df.shape}")

 new lamost gaisid matched dr11 dataframe shape =  (968, 76)


We'll now modufy this new lamost gaia matched dataframe to include just the columns we want. We will check against our earlier created dictionary of all the columns to ensure we ahve the corrcet column labels.

In [161]:
cols_to_keep  = ["gaia_source_id", "obsid", "uid", "designation", "gp_id","teff_lasp", "teff_lasp_err","logg_lasp", "logg_lasp_err", "feh_lasp", "feh_lasp_err", "vsini_lasp", "vsini_lasp_err","rv_lasp0", "rv_lasp0_err","rv_lasp1", "rv_lasp1_err", 'obsdate',  'ra', 'dec', 'snr', 'gaia_g_mean_mag', 'gaia_bp_mean_mag', 'gaia_rp_mean_mag']

valid_cols = [c for c in cols_to_keep if c in lamost_col_dtype_info_dict]
missing_cols = [c for c in cols_to_keep if c not in lamost_col_dtype_info_dict]

print("Valid columns:", valid_cols)
print("Missing columns:", missing_cols)

lamost_gaiaid_matched_dr11_df_2 = lamost_gaiaid_matched_dr11_df [valid_cols].copy()
print(lamost_gaiaid_matched_dr11_df_2.shape)
lamost_gaiaid_matched_dr11_df_2.head()



Valid columns: ['gaia_source_id', 'obsid', 'uid', 'designation', 'gp_id', 'teff_lasp', 'teff_lasp_err', 'logg_lasp', 'logg_lasp_err', 'feh_lasp', 'feh_lasp_err', 'vsini_lasp', 'vsini_lasp_err', 'rv_lasp0', 'rv_lasp0_err', 'rv_lasp1', 'rv_lasp1_err', 'obsdate', 'ra', 'dec', 'snr', 'gaia_g_mean_mag', 'gaia_bp_mean_mag', 'gaia_rp_mean_mag']
Missing columns: []
(968, 24)


,gaia_source_id,obsid,uid,designation,gp_id,teff_lasp,teff_lasp_err,logg_lasp,logg_lasp_err,feh_lasp,...,rv_lasp0_err,rv_lasp1,rv_lasp1_err,obsdate,ra,dec,snr,gaia_g_mean_mag,gaia_bp_mean_mag,gaia_rp_mean_mag
2893,2080605823183999616,590903018,b'G14076823824667',b'J194340.95+472654.9',2080605823183999616,5473.189941,45.490002,3.543,0.055,-0.020,...,0.96,-0.810000,1.13,b'2017-10-03',295.920650,47.448608,31.200001,13.25180,13.63460,12.70210
4072,3698079992771142784,635002202,b'G10997526900410',b'J121824.52-003304.8',3698079992771142784,5364.370117,67.809998,4.217,0.085,0.304,...,1.34,-14.460000,1.41,b'2018-01-25',184.602205,-0.551349,19.879999,14.13470,14.52380,13.58540
4239,2134719834131370496,590903146,b'G14059148581261',b'J194235.69+482944.0',2134719834131370496,5886.859863,38.540001,4.319,0.048,-0.131,...,0.80,-1.610000,1.00,b'2017-10-03',295.648710,48.495560,25.110001,12.76260,13.07190,12.26890
6803,2086419903235549312,590904106,b'G14058703644516',b'J194918.93+474855.6',2086419903235549312,6132.020020,34.619999,4.193,0.042,0.011,...,0.72,-18.629999,0.95,b'2017-10-03',297.328890,47.815472,27.520000,13.33180,13.62000,12.88710
17314,875071278432954240,611716080,b'G16235287266250',b'J074955.06+272147.8',875071278432954240,5714.950195,-9999.000000,4.557,-9999.000,-0.012,...,-9999.00,-16.000000,9999.00,b'2017-12-04',117.479439,27.363291,815.119995,6.73718,7.07632,6.22282


we will save the dataframe as a fit file, which can be imported in case the file needs to be run, as the downlaod file is too big and takes too logn to download.

In [163]:
fits_file = f"{path}lamost_dr11_gaia_matched_filtered.fits"

# using the astrpy package tp savbe dataframe as a table first
lahost_tbl = Table.from_pandas(lamost_gaiaid_matched_dr11_df_2)
# write FITS
lahost_tbl.write(fits_file, overwrite=True)

print(f" saved gaia id dr11 matched dataframe to FITS file:", fits_file)

 saved gaia id dr11 matched dataframe to FITS file: ./lamost_dr11_gaia_matched_filtered.fits


In [164]:
lamost_gaiaid_matched_dr11_df_2.shape

(968, 24)

.......Thsi is end of improtuing data sets..... have 4 dataframes

- stallar hosts = stellar_host_df > (46870, 136)
- pscompars = pscompars_df  >  (6053, 683)
- gaia DR3 = gaia_dr3_ap_df > (37431, 12)
- LAMOST = lamost_gaiaid_matched_dr11_df_2 > (968, 24)

In [ ]:
SELECT 
    c.table_name,
    c.column_name,
    c.datatype,
    c.unit,
    c.ucd,
    c.description
FROM tap_schema.columns AS c
WHERE c.table_name = 'public.stellar'
ORDER BY c.ordinal_position;


SELECT 
    c.table_name,
    c.column_name,
    c.datatype,
    c.unit,
    c.ucd,
    c.description
FROM tap_schema.columns AS c
WHERE c.table_name = 'public.catalogue'


SELECT column_name, datatype, description
FROM tap_schema.columns
WHERE table_name = 'public.med_stellar'



In [ ]:
# list of all DR10 colmn mames
obsid
uid
gp_id
designation
obsdate
lmjd
mjd
planid
spid
fiberid
ra_obs
dec_obs
snru
snrg
snrr
snri
snrz
class
subclass
z
z_err
ps_id
mag_ps_g
mag_ps_r
mag_ps_i
mag_ps_z
mag_ps_y
gaia_source_id
gaia_g_mean_mag
fibertype
offsets
offsets_v
ra
dec
fibermask
with_norm_flux


In [ ]:
lamost_dr10_schema = {
    "obsid": "Unique spectrum ID for each observation",
    "uid": "Unique source identifier (computed using ura + udec)",
    "gp_id": "Source ID from the original survey (Gaia, Pan-STARRS, or LAMOST)",
    "designation": "Target designation",
    "obsdate": "Observation date",
    "lmjd": "Local modified Julian day",
    "mjd": "Modified Julian day",
    "planid": "Plan name",
    "spid": "Spectrograph ID",
    "fiberid": "Fiber ID",
    "ra_obs": "Fiber pointing RA (deg)",
    "dec_obs": "Fiber pointing Dec (deg)",
    "snru": "S/N in u band",
    "snrg": "S/N in g band",
    "snrr": "S/N in r band",
    "snri": "S/N in i band",
    "snrz": "S/N in z band",
    "class": "Spectral type",
    "subclass": "Stellar subclass",
    "z": "Redshift",
    "z_err": "Redshift uncertainty",
    "ps_id": "Pan-STARRS objID",
    "mag_ps_g": "PS g-band magnitude",
    "mag_ps_r": "PS r-band magnitude",
    "mag_ps_i": "PS i-band magnitude",
    "mag_ps_z": "PS z-band magnitude",
    "mag_ps_y": "PS y-band magnitude",
    "gaia_source_id": "Gaia DR3 source_id",
    "gaia_g_mean_mag": "Gaia DR3 G-band magnitude",
    "fibertype": "Fiber type (Obj, Sky, F-std, etc.)",
    "offsets": "Flag for fiber offsets",
    "offsets_v": "Offset distance (arcsec)",
    "ra": "Input catalog RA (deg)",
    "dec": "Input catalog Dec (deg)",
    "fibermask": "Fiber problem flag",
    "with_norm_flux": "1 if normalized spectrum is included"
}


In [104]:
import os
import requests
import pandas as pd

# ----------------------
# 1. Download DR10 table
# ----------------------
url = "https://www.lamost.org/dr10/v2.0/catalogue/catalogue.csv"
out_file = "lamost_dr10_v2_catalogue.csv"

if not os.path.exists(out_file):
    print("Downloading LAMOST DR10 v2.0 catalogue...")
    r = requests.get(url, stream=True)
    open(out_file, "wb").write(r.content)
    print("Saved:", out_file)
else:
    print("File already exists:", out_file)

# Load into Pandas
df_lamost = pd.read_csv(out_file)
print("LAMOST shape:", df_lamost.shape)
df_lamost.head()


Saved: lamost_dr10_v2_catalogue.csv
LAMOST shape: (0, 2)


,"{""description"":""404 Not Found: The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.""","error:""Not Found""}"


In [107]:
import requests
import pandas as pd
import json

# LAMOST catalogue API endpoint
url2 = "https://www.lamost.org/dr10/ajax/catalogue"

payload = {
    "action": "export",               # tells LAMOST to export data
    "format": "csv",                  # csv output
    "limit": 0,                       # 0 means "all rows"
    "columns": "all",                 # all columns
    "filters": {},                    # no filters
}

print("Submitting request...")
r = requests.post(url2, data={"request": json.dumps(payload)})

# The server returns a download URL
download_url = r.json()["url"]
print("Download URL:", download_url2)


Submitting request...


KeyError: 'url'

In [108]:
import requests, json

url = "https://www.lamost.org/dr10/ajax/catalogue"
payload = {
    "action": "export",
    "format": "csv",
    "limit": 0,
    "columns": "all",
    "filters": {}
}

r = requests.post(url, data={"request": json.dumps(payload)})

print("Status:", r.status_code)
print("Raw text response:\n", r.text)


Status: 200
Raw text response:
 {"description":"404 Not Found: The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.","error":"Not Found"}



In [109]:
import requests

url = "http://dr10.lamost.org/v1/catalogs/DR10/lamost_dr10_v1_stellar.fits"
outfile = "lamost_dr10_stellar_dr10.fits"

print("Downloading LAMOST DR10 Stellar FITS...")
r = requests.get(url, stream=True)

with open(outfile, "wb") as f:
    f.write(r.content)

print("Saved:", outfile)


ConnectionError: HTTPConnectionPool(host='dr10.lamost.org', port=80): Max retries exceeded with url: /v1/catalogs/DR10/lamost_dr10_v1_stellar.fits (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x00000157D0F8F740>: Failed to resolve 'dr10.lamost.org' ([Errno 11001] getaddrinfo failed)"))

In [ ]:
# functin to map column mnames
# will create a new column name Year at the start and place 2016 in each row for it
def gaia_dr3_ap_rename_cols(df):
    df = df.rename(columns={
        'source_id': 'country',
        'teff_gspphot': 'region',
        'logg_gspphot': 'happiness_rank',
        'mg_gspphot': 'happiness_score',
        'mh_gspphot': 'economy(gdp_per_capita)',
        'radius_gspphot': 'family',
        'distance_gspphot': 'health(life_exp)',
        'lum_flame': 'freedom',
        'mass_flame': 'trust(government_corruption)',
        'age_flame': 'generosity',
        'evolstage_flame': 'dystopia_residual'
        'ag_gspphot':
        
        
            SELECT source_id, teff_gspphot, logg_gspphot, mg_gspphot, mh_gspphot, radius_gspphot, distance_gspphot, lum_flame, mass_flame, age_flame, evolstage_flame, ag_gspphot
    })
    df = df.drop(columns=['Lower Confidence Interval', 'Upper Confidence Interval', 'region', 'dystopia_residual'], errors='ignore')
    df['year'] = 2016
    return df